In [ ]:
# =========================================================
# LeJEPA — YOLOv8 Backbone (Full-Image Augmentations)
# Fully Commented Line-by-Line Script
#
# LeJEPA Total Loss Formula:
#    LeJEPA Loss = (1 - λ) * L_pred + λ * SIGReg
#
# Key Features:
#   1. Backbone: YOLOv8 convolutional backbone (layers 0–9).
#   2. Augmentations: Purely geometric transformations (no color jitter).
#   3. Hardware: DirectML support for Windows/Dell AMD/NVIDIA GPUs with CUDA/CPU fallbacks.
#   4. Tracking: Saves 3D PCA interactive HTML plots at epochs 1, 3, and 120.
# =========================================================

# Import standard library for OS interaction and filesystem paths
import os  # System operations (directory creation, path manipulation)
from pathlib import Path  # Object-oriented filesystem path handler
from PIL import Image  # Pillow image processing library to open RGB images

# Import numerical computation and data analysis libraries
import numpy as np  # Array processing and matrix operations
import pandas as pd  # Dataframe handling for saving training metrics to CSV

# Import PyTorch deep learning framework and core sub-modules
import torch  # Core PyTorch tensor library
import torch.nn as nn  # Neural network layer modules (Linear, Conv2d, Sequential)
from torch.utils.data import Dataset, DataLoader  # Data loading utilities

# Import progress bar and computer vision transformation tools
import tqdm  # Progress bar display for training loops
from torchvision.transforms import v2  # torchvision v2 image transformation transforms
from torchvision.ops import MLP  # Multi-Layer Perceptron utility block

# Import dimensionality reduction and interactive visualization libraries
from sklearn.decomposition import PCA  # Principal Component Analysis for 3D embeddings
import plotly.express as px  # High-level Plotly API for interactive 3D scatter plots
import plotly.graph_objects as go  # Low-level Plotly API for custom loss curves

# Safely import the Ultralytics YOLO package for feature backbone extraction
try:
    from ultralytics import YOLO  # Import official YOLO class from Ultralytics
except ImportError:
    # Raise clear error if the package is missing from the environment
    raise ImportError("Please install ultralytics package via: pip install ultralytics")

# Safely import DirectML for Windows / Dell GPU acceleration (AMD, Intel, NVIDIA)
try:
    import torch_directml  # Import DirectML PyTorch adapter
    DEVICE = torch_directml.device()  # Set device to DirectML GPU accelerator
    print(f"⚡ Using DirectML device: {DEVICE}")  # Notify user of active DirectML backend
    USE_AMP = False  # Disable Automatic Mixed Precision (AMP) as DirectML runs in FP32
except ImportError:
    # Fall back to native CUDA GPU if available, otherwise CPU
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select CUDA or CPU
    USE_AMP = True if DEVICE.type == "cuda" else False  # Enable AMP only if CUDA is active
    print(f"ℹ️ torch_directml not found. Using device: {DEVICE}")  # Notify active PyTorch device


# =========================================================
# CONFIGURATION & HYPERPARAMETERS
# =========================================================

# Directory path containing the multi-camera pre-training dataset
JPEG_ROOT = r"C:\Users\CLIMDES LAB PC1\Downloads\SSL  FOR COTTON FIELD\Joint pretraining data"

# Directory path where model checkpoints, CSV logs, and HTML plots will be saved
CHECKPOINT_DIR = r"C:\Users\CLIMDES LAB PC1\Downloads\SSL  FOR COTTON FIELD\LeJEPA_YoloV8_Pretraining_Checkpoints"

IMAGE_SIZE = 128  # Target image height and width (H=128, W=128) in pixels
BATCH_SIZE = 8  # Number of samples (images) processed in each training batch
EPOCHS = 120  # Total number of pre-training epochs
V = 2  # Number of augmented views generated per single image sample

PROJ_DIM = 128  # Output dimension of the LeJEPA MLP projector network
LR = 0.001788039218013555  # Peak learning rate for AdamW optimizer
LR_MIN = 1e-4  # Minimum learning rate target at the end of cosine annealing decay
WEIGHT_DECAY = 0.005291842753172914  # L2 penalty factor for AdamW weight decay

# Trade-off parameter λ weighting Prediction Loss (1 - λ) vs SIGReg Regularization (λ)
LAMBDA = 0.5

NUM_WORKERS = 0  # DataLoader sub-processes (0 means main thread loads data)
PIN_MEMORY = False  # Set True if using CUDA host memory pinning for faster transfers

# Set target epochs for printing embeddings and generating 3D PCA interactive HTML plots
# Epochs 1, 3, and 120 correspond to 0-indexed values: 0, 2, and 119
PCA_EPOCHS = {0, 2, EPOCHS - 1}


# =========================================================
# SIGREG REGULARIZATION MODULE
# =========================================================
class SIGReg(nn.Module):
    """
    Skew-Invariant Gaussian Regularization (SIGReg) module.
    Prevents representation collapse by penalizing deviation from an isotropic distribution
    using a characteristic function integration over random projections.
    """
    def __init__(self, knots=17):
        super().__init__()  # Initialize PyTorch parent nn.Module class
        
        t = torch.linspace(0, 3, knots)  # Create 17 evenly spaced evaluation points between 0 and 3
        dt = 3 / (knots - 1)  # Calculate uniform step size between consecutive knots
        
        weights = torch.full((knots,), 2 * dt)  # Initialize trapezoidal integration weights array
        weights[[0, -1]] = dt  # Adjust boundary weights for numerical integration
        
        window = torch.exp(-t.square() / 2.0)  # Evaluate Gaussian window envelope exp(-t^2 / 2)

        # Register non-trainable buffers so they automatically move to GPU/DirectML device with the module
        self.register_buffer("t", t)  # Integration evaluation points buffer
        self.register_buffer("phi", window)  # Gaussian target characteristic window buffer
        self.register_buffer("weights", weights * window)  # Pre-computed integration weights buffer

    def forward(self, proj):
        """
        Forward pass for SIGReg computation.
        Args:
            proj (torch.Tensor): Projected embeddings of shape (V, B, D) or (B, D).
        Returns:
            torch.Tensor: Scalar SIGReg loss value.
        """
        # If input tensor is 2D (B, D), insert a view dimension to make it 3D (1, B, D)
        if proj.dim() == 2:
            proj = proj.unsqueeze(0)  # Shape becomes (1, B, D)

        Vv, B, D = proj.shape  # Unpack view count (Vv), batch size (B), projection dimension (D)

        # Generate a random Gaussian projection matrix A of shape (D, 256) on the same device
        A = torch.randn(D, 256, device=proj.device)  # Random direction vectors
        A = A / (A.norm(dim=0, keepdim=True) + 1e-12)  # Unit-normalize projection columns

        # Project representations onto random directions and multiply by grid points t
        x_t = (proj @ A).unsqueeze(-1) * self.t  # Tensor shape: (V, B, 256, knots)

        # Calculate deviation of empirical characteristic function from theoretical Gaussian target
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()  # Squared error
        
        # Integrate squared error weighted by Simpson/trapezoidal weights scaled by batch size
        statistic = (err @ self.weights) * B  # Compute sample statistic per projection
        
        return statistic.mean()  # Return average SIGReg loss across all projection directions


# =========================================================
# DATASET MODULE (Multi-Camera, Full-Image Geometric Views)
# =========================================================
class ForageDataset(Dataset):
    """
    Dataset loader that recursively collects images across multi-camera folder structures
    and generates two independently augmented geometric views per image sample.
    """
    def __init__(self, root):
        root = Path(root)  # Convert path string to Path object
        if not root.exists():  # Validate dataset directory existence
            raise FileNotFoundError(f"JPEG_ROOT not found: {root}")

        # Supported image extensions
        valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
        
        # Recursively search all subdirectories (handles multi-camera folders automatically)
        self.paths = [
            str(p) for p in root.rglob("*") if p.suffix.lower() in valid_exts
        ]
        
        if len(self.paths) == 0:  # Raise error if no matching images are found
            raise RuntimeError(f"No valid images found under dataset root: {root}")

        print(f"📸 Dataset loaded: {len(self.paths)} total images found across camera folders.")

        # Define pure geometric transformation pipeline (no color or brightness modifications)
        self.aug = v2.Compose([
            # Crop random image region (50%-100% scale) and resize back to IMAGE_SIZE x IMAGE_SIZE
            v2.RandomResizedCrop(IMAGE_SIZE, scale=(0.5, 1.0), ratio=(0.9, 1.1)),
            
            v2.RandomHorizontalFlip(p=0.5),  # 50% probability horizontal flip
            v2.RandomVerticalFlip(p=0.1),  # 10% probability vertical flip
            v2.RandomRotation(degrees=25),  # Rotate randomly between -25 and +25 degrees
            
            # Apply affine transform: shift image up to 8% horizontally/vertically, scale 85%-115%
            v2.RandomAffine(
                degrees=0,  # Rotation already handled above
                translate=(0.08, 0.08),  # Translation jitter ratio
                scale=(0.85, 1.15),  # Scale factor range
            ),
            
            v2.ToImage(),  # Convert PIL Image or numpy array to torchvision Image object
            v2.ToDtype(torch.float32, scale=True),  # Convert pixel integers (0-255) to float (0.0-1.0)
            
            # Normalize channel values using standard ImageNet mean and standard deviation
            v2.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.paths)  # Return total count of dataset images

    def __getitem__(self, idx):
        """
        Loads an image from disk and returns V independently augmented geometric views.
        """
        img_path = self.paths[idx]  # Get file path at index idx
        img = Image.open(img_path).convert("RGB")  # Open image and force 3-channel RGB mode
        
        # Apply the geometric augmentation pipeline V times independently to generate V views
        views = [self.aug(img) for _ in range(V)]  # List of tensor views, each (3, H, W)
        
        return torch.stack(views, dim=0)  # Stack views into a single tensor of shape (V, 3, H, W)


# =========================================================
# ENCODER MODULE (YOLOv8 Feature Backbone + MLP Projector)
# =========================================================
class YOLOv8Encoder(nn.Module):
    """
    Encoder wrapping the standard YOLOv8 convolutional feature extraction backbone
    coupled with an Adaptive Average Pooling layer and a 3-layer MLP projector.
    """
    def __init__(self, weights="yolov8n.pt", proj_dim=128):
        super().__init__()  # Initialize parent PyTorch module
        
        # Instantiates full YOLO model structure using Ultralytics
        yolo_model = YOLO(weights).model  # Access underlying PyTorch nn.Module instance
        
        # Slice layers 0 through 9 representing the complete feature extraction backbone of YOLOv8
        self.backbone = nn.Sequential(*list(yolo_model.model[:10]))
        
        self.pool = nn.AdaptiveAvgPool2d(1)  # Adaptive average pooling layer to shrink spatial dims to 1x1

        # Calculate exact output channel dimension dynamically using a dummy tensor forward pass
        with torch.no_grad():  # Disable gradient tracking during dummy inference
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)  # Create dummy input tensor
            feat_dim = self.pool(self.backbone(dummy)).flatten(1).shape[1]  # Extract channel count

        print(f"🧱 YOLOv8 Backbone loaded successfully | Feature embedding dimension: {feat_dim}")

        # Instantiate a 3-layer Multi-Layer Perceptron (MLP) projector with BatchNorm
        self.proj = MLP(
            feat_dim,  # Input layer dimension matching YOLOv8 backbone output features
            [2048, 2048, proj_dim],  # Hidden layer dimensions and final projection output dimension
            norm_layer=nn.BatchNorm1d  # Batch normalization applied between projector layers
        )

    def forward(self, x):
        """
        Forward pass for the encoder network.
        Args:
            x (torch.Tensor): Input batch tensor of shape (B, V, 3, H, W).
        Returns:
            z (torch.Tensor): Unprojected backbone embeddings of shape (B, V, feat_dim).
            p (torch.Tensor): Projected embeddings of shape (B, V, proj_dim).
        """
        B, Vv = x.shape[:2]  # Extract batch size B and view count Vv
        x = x.flatten(0, 1)  # Merge batch and view dimensions into shape (B * V, 3, H, W)

        z = self.backbone(x)  # Extract feature maps via YOLOv8 backbone -> shape (B*V, C, H_out, W_out)
        z = self.pool(z).flatten(1)  # Apply global average pooling -> shape (B*V, feat_dim)

        p = self.proj(z)  # Pass pooled backbone features through MLP projector -> shape (B*V, proj_dim)

        z = z.view(B, Vv, -1)  # Reshape backbone embeddings back to (B, V, feat_dim)
        p = p.view(B, Vv, -1)  # Reshape projector representations back to (B, V, proj_dim)
        
        return z, p  # Return both backbone embeddings and projected vectors


# =========================================================
# LeJEPA PREDICTION LOSS (L2 Invariance Loss)
# =========================================================
def lejepa_prediction_loss(proj: torch.Tensor) -> torch.Tensor:
    """
    Computes L2 prediction/invariance loss across multi-view projections.
    Measures the mean squared deviation of each view projection from the centroid (mean) view.
    
    Args:
        proj (torch.Tensor): Projected tensor of shape (B, V, D).
    Returns:
        torch.Tensor: Scalar prediction loss.
    """
    mu = proj.mean(dim=1, keepdim=True)  # Compute mean projection across views V -> shape (B, 1, D)
    dif = mu - proj  # Difference vector between centroid and view projections -> shape (B, V, D)
    return dif.square().mean()  # Return scalar mean squared error (ℓ2 norm)


# =========================================================
# LOGGING & VISUALIZATION UTILITIES
# =========================================================
@torch.no_grad()
def save_interactive_pca(net, loader, epoch, out_dir: Path, max_batches=60):
    """
    Generates 3D PCA projection scatter plot saved as an interactive HTML file
    and prints representative embedding stats for verification at target epochs.
    """
    net.eval()  # Set network to evaluation mode (disables dropout and batch norm updates)
    feats = []  # List to store extracted feature batches

    batches_seen = 0  # Counter for batches processed
    for vs in loader:  # Iterate over DataLoader batches
        vs = vs.to(DEVICE)  # Move input tensor batch to compute device -> shape (B, V, 3, H, W)
        emb, _ = net(vs)  # Extract backbone embeddings -> shape (B, V, feat_dim)
        emb = emb.reshape(-1, emb.shape[-1])  # Flatten to shape (B * V, feat_dim)
        feats.append(emb.detach().cpu().numpy())  # Transfer tensor to host memory and save as numpy array

        batches_seen += 1  # Increment batch counter
        if batches_seen >= max_batches:  # Stop after processing specified batch limit
            break

    X = np.concatenate(feats, axis=0)  # Concatenate batch numpy arrays into matrix (N_samples, feat_dim)

    # Print detailed embedding metrics to console
    print(f"\n--- [EMBEDDINGS INSPECTION - EPOCH {epoch + 1}] ---")
    print(f"Embeddings Matrix Shape: {X.shape}")
    print(f"Embeddings Mean Value: {X.mean():.4f} | Standard Deviation: {X.std():.4f}")
    print(f"Sample Embedding Values (Row 0, First 5 Features):\n{X[0, :5]}\n" + "-" * 55)

    # Perform 3-component Principal Component Analysis (PCA)
    Z = PCA(n_components=3, random_state=0).fit_transform(X)

    # Build 3D interactive scatter plot using Plotly Express
    fig = px.scatter_3d(
        x=Z[:, 0], y=Z[:, 1], z=Z[:, 2],  # Assign PCA components to X, Y, Z axes
        opacity=0.7,  # Set point opacity
        title=f"YOLOv8 Embeddings — Epoch {epoch + 1}",  # Plot title
    )
    fig.update_traces(marker=dict(size=3))  # Adjust marker point size
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))  # Adjust layout margins

    out_dir.mkdir(parents=True, exist_ok=True)  # Create destination folder if it doesn't exist
    out_path = out_dir / f"pca_epoch_{epoch+1:03d}.html"  # Construct output HTML filename
    fig.write_html(str(out_path), include_plotlyjs="cdn")  # Save interactive figure to disk
    print(f"🧩 Saved interactive 3D PCA plot: {out_path}")


def save_training_curves(history, out_dir: Path):
    """
    Saves interactive line chart showing total loss, L2 prediction loss,
    and SIGReg regularization loss over all training epochs.
    """
    out_dir.mkdir(parents=True, exist_ok=True)  # Ensure target directory exists
    epochs = np.arange(1, len(history["lejepa_total"]) + 1)  # Array of 1-indexed epoch numbers

    fig = go.Figure()  # Instantiate Plotly figure object
    
    # Add total LeJEPA loss trace
    fig.add_trace(go.Scatter(x=epochs, y=history["lejepa_total"], mode="lines", name="LeJEPA total"))
    
    # Add prediction / invariance (L2) loss trace
    fig.add_trace(go.Scatter(x=epochs, y=history["pred"], mode="lines", name="Prediction / invariance (L2)"))
    
    # Add SIGReg regularization loss trace
    fig.add_trace(go.Scatter(x=epochs, y=history["sigreg"], mode="lines", name="SIGReg"))

    # Configure chart title, labels, hover mode, and layout bounds
    fig.update_layout(
        title="Training Loss Curves — YOLOv8 Backbone",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=60, b=40),
    )

    out_path = out_dir / "training_losses.html"  # Output file path
    fig.write_html(str(out_path), include_plotlyjs="cdn")  # Export HTML interactive figure
    print(f"📉 Saved interactive training loss plot: {out_path}")


def save_loss_history(history, out_dir: Path, model_name: str):
    """
    Saves numerical training loss records to a CSV file for future comparison.
    """
    out_dir.mkdir(parents=True, exist_ok=True)  # Create output directory path if missing

    # Assemble pandas DataFrame containing epoch-wise loss history
    df = pd.DataFrame({
        "epoch": np.arange(1, len(history["lejepa_total"]) + 1),
        "prediction_loss": history["pred"],
        "sigreg_loss": history["sigreg"],
        "lejepa_total": history["lejepa_total"],
    })

    out_path = out_dir / f"loss_history_{model_name}.csv"  # File path construction
    df.to_csv(out_path, index=False)  # Write DataFrame to CSV without row index
    print(f"📁 Saved loss history CSV file: {out_path}")


# =========================================================
# MAIN TRAINING EXECUTION LOOP
# =========================================================
def main():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)  # Ensure primary checkpoint path exists
    ckpt_dir = Path(CHECKPOINT_DIR)  # Convert path string to Path object

    # Define sub-directories for saving plots and PCA exports
    plots_dir = ckpt_dir / "YOLOv8_interactive_plots" / "yolov8"
    pca_dir = plots_dir / "pca_3d"

    torch.manual_seed(0)  # Seed PyTorch random number generator for reproducibility

    dataset = ForageDataset(JPEG_ROOT)  # Instantiate dataset loader
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,  # Mini-batch size
        shuffle=True,  # Shuffle data order every epoch
        drop_last=True,  # Drop last incomplete batch to ensure stable tensor shapes
        num_workers=NUM_WORKERS,  # Worker thread count
        pin_memory=PIN_MEMORY  # Memory pinning flag
    )

    # Instantiate YOLOv8 Encoder model and move parameters to active device
    net = YOLOv8Encoder(weights="yolov8n.pt", proj_dim=PROJ_DIM).to(DEVICE)
    
    # Instantiate SIGReg loss module and move buffer parameters to device
    sigreg = SIGReg().to(DEVICE)

    # Initialize AdamW optimizer over encoder model parameters
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    warmup = len(loader)  # Warmup steps equivalent to 1 full epoch of batch iterations
    total = len(loader) * EPOCHS  # Total training iterations across all epochs
    
    # Construct learning rate scheduler combining Linear Warmup and Cosine Annealing Decay
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        schedulers=[
            # Linear LR warmup for the first epoch
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
            # Cosine decay down to LR_MIN for the remaining steps
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, total - warmup), eta_min=LR_MIN),
        ],
        milestones=[warmup]  # Step milestone switching from linear warmup to cosine decay
    )

    # Gradient Scaler for Automatic Mixed Precision (AMP) when running on CUDA
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    # Dictionary to record epoch-averaged loss histories
    history = {"lejepa_total": [], "pred": [], "sigreg": []}

    # Loop through all training epochs
    for epoch in range(EPOCHS):
        net.train()  # Set network model to training mode

        ep_total = 0.0  # Accumulator for total batch losses
        ep_pred = 0.0  # Accumulator for prediction losses
        ep_sig = 0.0  # Accumulator for SIGReg losses
        n_batches = 0  # Counter for batch iterations

        pbar = tqdm.tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")  # Initialize tqdm progress bar
        for vs in pbar:  # Batch iteration loop
            vs = vs.to(DEVICE)  # Move input images tensor to hardware compute device -> shape (B, V, 3, H, W)

            # Mixed precision context manager
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                _, proj = net(vs)  # Extract projections -> shape (B, V, PROJ_DIM)
                
                pred_loss = lejepa_prediction_loss(proj)  # Compute L2 invariance prediction loss
                sig_loss = sigreg(proj.transpose(0, 1))  # Compute SIGReg regularization loss over shape (V, B, D)

                # Combine losses using trade-off hyperparameter λ
                loss = (1.0 - LAMBDA) * pred_loss + LAMBDA * sig_loss

            opt.zero_grad(set_to_none=True)  # Reset parameter gradients to None for memory efficiency
            
            if USE_AMP:
                scaler.scale(loss).backward()  # Compute scaled backpropagation gradients
                scaler.step(opt)  # Step optimizer weights using scaled gradients
                scaler.update()  # Update AMP gradient scale factor
            else:
                loss.backward()  # Standard backpropagation
                opt.step()  # Standard optimizer step

            sched.step()  # Advance learning rate scheduler step

            # Accumulate loss terms
            ep_total += float(loss.detach().cpu())
            ep_pred += float(pred_loss.detach().cpu())
            ep_sig += float(sig_loss.detach().cpu())
            n_batches += 1  # Increment processed batch count

        # Calculate epoch averages
        ep_total /= max(1, n_batches)
        ep_pred /= max(1, n_batches)
        ep_sig /= max(1, n_batches)

        # Append epoch statistics to history records
        history["lejepa_total"].append(ep_total)
        history["pred"].append(ep_pred)
        history["sigreg"].append(ep_sig)

        # Print current epoch loss metrics to console
        print(
            f"Epoch {epoch+1:03d} | "
            f"LeJEPA Total Loss: {ep_total:.6f} | "
            f"Pred Loss (L2): {ep_pred:.6f} | "
            f"SIGReg Loss: {ep_sig:.6f}"
        )

        # Trigger embedding inspection and 3D PCA plot creation on specified epochs (Epochs 1, 3, and 120)
        if epoch in PCA_EPOCHS:
            save_interactive_pca(net, loader, epoch, pca_dir)

    # Save finalized model checkpoint weights
    ckpt_path = ckpt_dir / "lejepa_yolov8_fullimage_L2pred.pth"
    torch.save(net.state_dict(), ckpt_path)  # Save parameter state dict to file
    print(f"🖤 Model checkpoint saved successfully to: {ckpt_path}")

    # Save visual training loss curves HTML plot and loss history CSV file
    save_training_curves(history, plots_dir)
    save_loss_history(history, plots_dir, model_name="yolov8")


# Standard Python script entry point
if __name__ == "__main__":
    main()  # Run main pre-training function

⚡ Using DirectML device: privateuseone:0
📸 Dataset loaded: 355008 total images found across camera folders.
🧱 YOLOv8 Backbone loaded successfully | Feature embedding dimension: 256


Epoch 1/120:   0%|          | 0/44376 [00:00<?, ?it/s]C:\Users\CLIMDES LAB PC1\AppData\Local\Temp\ipykernel_30252\1315496148.py:140: UserWarning: The operator 'aten::addmv.out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  statistic = (err @ self.weights) * B  # Compute sample statistic per projection
Epoch 1/120:  73%|███████▎  | 32263/44376 [1:17:31<30:15,  6.67it/s]  

In [2]:
%pip install ultralytics

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.4 MB 1.8 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 2.0 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.4 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/846.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/846.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.8 kB ? eta -:--:--
   ------------ --------------------------- 262.1/846.8 kB ? eta -:--:--
   ----------------------- -------------- 524.3/846.8 kB 762.0 kB/s eta 0:00:01
   ---------------------------------------- 846.8/846.8 kB 849.9 kB/s  0:00:00
   ---------------------------------------- 0.0/52.6 MB ? eta -:--:--
   ---------------------------------------- 